<a href="https://colab.research.google.com/github/goitstudent123/numerical_programming_python/blob/main/%D0%94%D0%9710_%D0%93%D0%90%D0%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1 — Load the Iris dataset
from sklearn.datasets import load_iris
import numpy as np
import pandas as pd

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Classes:", target_names)

X shape: (150, 4)
y shape: (150,)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: ['setosa' 'versicolor' 'virginica']


In [2]:
# Step 2 — Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Class distribution (train):", dict(zip(*np.unique(y_train, return_counts=True))))
print("Class distribution (test):", dict(zip(*np.unique(y_test, return_counts=True))))

Train: (105, 4) (105,)
Test: (45, 4) (45,)
Class distribution (train): {np.int64(0): np.int64(35), np.int64(1): np.int64(35), np.int64(2): np.int64(35)}
Class distribution (test): {np.int64(0): np.int64(15), np.int64(1): np.int64(15), np.int64(2): np.int64(15)}


In [3]:
# Step 3 — Feature samples per class (from training set)
classes = np.unique(y_train)
X_train_by_class = {c: X_train[y_train == c] for c in classes}

for c in classes:
    print(f"class {c} ({iris.target_names[c]}):", X_train_by_class[c].shape)

class 0 (setosa): (35, 4)
class 1 (versicolor): (35, 4)
class 2 (virginica): (35, 4)


In [4]:
# Step 4 — Covariance matrices per class (ddof=1)
cov_by_class = {}
mean_by_class = {}

for c in classes:
    Xc = X_train_by_class[c]
    mean_by_class[c] = Xc.mean(axis=0)
    cov_by_class[c] = np.cov(Xc, rowvar=False, ddof=1)

for c in classes:
    print(f"Class {c} mean:", mean_by_class[c])
    print(f"Class {c} covariance shape:", cov_by_class[c].shape)

Class 0 mean: [4.98857143 3.42571429 1.48571429 0.24      ]
Class 0 covariance shape: (4, 4)
Class 1 mean: [5.94857143 2.73142857 4.23714286 1.30857143]
Class 1 covariance shape: (4, 4)
Class 2 mean: [6.68285714 3.00857143 5.63142857 2.06857143]
Class 2 covariance shape: (4, 4)


In [5]:
# Step 5 — Inverse covariance matrices (pinv fallback, tiny regularization if needed)
inv_cov_by_class = {}
logdet_by_class = {}

for c in classes:
    cov = cov_by_class[c]
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        cov = cov + 1e-9 * np.eye(cov.shape[0])
        sign, logdet = np.linalg.slogdet(cov)
    inv_cov_by_class[c] = np.linalg.pinv(cov)
    logdet_by_class[c] = logdet

for c in classes:
    print(f"Class {c} log|Σ|:", logdet_by_class[c])

Class 0 log|Σ|: -13.590604368258031
Class 1 log|Σ|: -10.70800763828154
Class 2 log|Σ|: -8.897090092582586


In [6]:
# Step 6 — Class priors from training labels
counts = np.bincount(y_train, minlength=len(classes))
priors = counts / counts.sum()

for c in classes:
    print(f"Prior π_{c}:", priors[c])

Prior π_0: 0.3333333333333333
Prior π_1: 0.3333333333333333
Prior π_2: 0.3333333333333333


In [7]:
# Step 7 — Single-vector discriminant (QDA)
# g_k(x) = -0.5*log|Σ_k| - 0.5*(x-μ_k)^T Σ_k^{-1} (x-μ_k) + log π_k

def qda_discriminant_single(x, mean_by_class, inv_cov_by_class, logdet_by_class, priors, classes):
    scores = {}
    for c in classes:
        mu = mean_by_class[c]
        inv_cov = inv_cov_by_class[c]
        logdet = logdet_by_class[c]
        delta = x - mu
        quad = delta @ inv_cov @ delta.T
        scores[c] = -0.5 * logdet - 0.5 * quad + np.log(priors[c])
    return scores

scores_example = qda_discriminant_single(
    X_test[0], mean_by_class, inv_cov_by_class, logdet_by_class, priors, classes
)
print("Single x scores:", scores_example)


Single x scores: {np.int64(0): np.float64(-644.1548068508239), np.int64(1): np.float64(-6.997763490762359), np.int64(2): np.float64(1.3653346383445297)}


In [8]:
# Step 8 — Full-matrix discriminants and probabilities
def qda_predict_proba(X, mean_by_class, inv_cov_by_class, logdet_by_class, priors, classes):
    G = np.zeros((X.shape[0], len(classes)))
    for idx, c in enumerate(classes):
        mu = mean_by_class[c]
        inv_cov = inv_cov_by_class[c]
        logdet = logdet_by_class[c]
        delta = X - mu
        quad = np.einsum('ij,jk,ik->i', delta, inv_cov, delta)
        G[:, idx] = -0.5 * logdet - 0.5 * quad + np.log(priors[c])
    G_max = G.max(axis=1, keepdims=True)
    expG = np.exp(G - G_max)
    probs = expG / expG.sum(axis=1, keepdims=True)
    preds = classes[probs.argmax(axis=1)]
    return preds, probs, G

y_pred_custom, proba_custom, G_custom = qda_predict_proba(
    X_test, mean_by_class, inv_cov_by_class, logdet_by_class, priors, classes
)

print("Pred shape:", y_pred_custom.shape)
print("Proba shape:", proba_custom.shape)
print("First 5 preds:", y_pred_custom[:5])
print("First 5 probs:\n", np.round(proba_custom[:5], 4))


Pred shape: (45,)
Proba shape: (45, 3)
First 5 preds: [2 1 1 1 2]
First 5 probs:
 [[0.000e+00 2.000e-04 9.998e-01]
 [0.000e+00 9.923e-01 7.700e-03]
 [0.000e+00 7.679e-01 2.321e-01]
 [0.000e+00 9.913e-01 8.700e-03]
 [0.000e+00 1.700e-01 8.300e-01]]


In [9]:
# Step 9 — sklearn QDA and comparison
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

qda = QuadraticDiscriminantAnalysis(store_covariance=True)
qda.fit(X_train, y_train)

y_pred_sklearn = qda.predict(X_test)
proba_sklearn = qda.predict_proba(X_test)

acc_custom = accuracy_score(y_test, y_pred_custom)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)
cm_custom = confusion_matrix(y_test, y_pred_custom, labels=classes)
cm_sklearn = confusion_matrix(y_test, y_pred_sklearn, labels=classes)

print("Custom QDA accuracy:", round(acc_custom, 4))
print("Sklearn QDA accuracy:", round(acc_sklearn, 4))
print("\nCustom confusion matrix (rows=true, cols=pred):\n", cm_custom)
print("\nsklearn confusion matrix (rows=true, cols=pred):\n", cm_sklearn)

print("\nCustom classification report:\n", classification_report(y_test, y_pred_custom, target_names=target_names))
print("sklearn classification report:\n", classification_report(y_test, y_pred_sklearn, target_names=target_names))


Custom QDA accuracy: 0.9778
Sklearn QDA accuracy: 0.9778

Custom confusion matrix (rows=true, cols=pred):
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

sklearn confusion matrix (rows=true, cols=pred):
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

Custom classification report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.94      1.00      0.97        15
   virginica       1.00      0.93      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

sklearn classification report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.94      1.00      0.97        15
   virginica       1.00      0.93      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        

In [10]:
# Step 10 — Metrics for similarity (numbers only; paste them back to me later)
from scipy.spatial.distance import cosine

agreement = np.mean(y_pred_custom == y_pred_sklearn)
cos_sims = [1 - cosine(proba_custom[i], proba_sklearn[i]) for i in range(len(X_test))]
avg_cos_sim = float(np.mean(cos_sims))
l1_dist = float(np.mean(np.abs(proba_custom - proba_sklearn)))
l2_dist = float(np.sqrt(np.mean((proba_custom - proba_sklearn) ** 2)))

side_by_side = pd.DataFrame({
    "y_true": y_test,
    "pred_custom": y_pred_custom,
    "pred_sklearn": y_pred_sklearn
})
proba_cols_custom = [f"p_custom_{name}" for name in target_names]
proba_cols_sklearn = [f"p_sklearn_{name}" for name in target_names]
proba_df = pd.DataFrame(
    np.hstack([proba_custom, proba_sklearn]),
    columns=proba_cols_custom + proba_cols_sklearn
)
preview = pd.concat([side_by_side, proba_df], axis=1).head(10)

print("=== Similarity metrics ===")
print("Accuracy (custom):", round(float((y_pred_custom == y_test).mean()), 6))
print("Accuracy (sklearn):", round(float((y_pred_sklearn == y_test).mean()), 6))
print("Prediction agreement rate:", round(agreement, 6))
print("Avg cosine similarity (probabilities):", round(avg_cos_sim, 6))
print("Mean L1 distance (probabilities):", round(l1_dist, 8))
print("RMSE (probabilities):", round(l2_dist, 8))

print("\n=== First 10 rows (truth, preds, probabilities) ===")
print(preview.to_string(index=False))


=== Similarity metrics ===
Accuracy (custom): 0.977778
Accuracy (sklearn): 0.977778
Prediction agreement rate: 1.0
Avg cosine similarity (probabilities): 1.0
Mean L1 distance (probabilities): 0.0
RMSE (probabilities): 0.0

=== First 10 rows (truth, preds, probabilities) ===
 y_true  pred_custom  pred_sklearn  p_custom_setosa  p_custom_versicolor  p_custom_virginica  p_sklearn_setosa  p_sklearn_versicolor  p_sklearn_virginica
      2            2             2    4.508824e-281         2.332659e-04        9.997667e-01     4.508824e-281          2.332659e-04         9.997667e-01
      1            1             1    5.800700e-126         9.923224e-01        7.677627e-03     5.800700e-126          9.923224e-01         7.677627e-03
      2            1             1    1.260934e-158         7.679236e-01        2.320764e-01     1.260934e-158          7.679236e-01         2.320764e-01
      1            1             1    1.920073e-129         9.913174e-01        8.682611e-03     1.920073e-12

# Conclusion

Manual QDA implementation matched sklearn essentially perfectly.

Key points:

* Both achieved identical accuracy (0.9778).
* Confusion matrices are identical.
* Prediction agreement rate is 1.0 — there was not a single disagreement.
* Average cosine similarity between probability vectors = 1.0
* L1 distance = 0.0
* RMSE = 0.0

This means the discriminant you wrote reproduces sklearn’s math bit-by-bit.

In this experiment QDA implementation is validated as fully correct and numerically identical to sklearn’s implementation on the Iris dataset.

This is the expected theoretical result when priors, covariance estimation, and logdet formulation are aligned exactly.